In [1]:
import os
import json
import shutil
import logging
from pathlib import Path
from typing import List, Tuple
from urllib.parse import urlparse

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('filter_podcast_urls.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


In [2]:
# Configuration
SOURCE_FOLDER = "apollo_podcast"  # Source folder containing JSON files
DESTINATION_FOLDER = "apollo_podcast_urls"  # Destination folder for files with podcast URLs

# Create destination folder if it doesn't exist
os.makedirs(DESTINATION_FOLDER, exist_ok=True)


In [3]:
def is_podcast_url(url: str) -> bool:
    """
    Check if a URL is podcast-related using multiple patterns.
    
    Patterns checked:
    1. Contains 'podcast' keyword (case-insensitive)
    2. Is from a known podcast platform
    """
    url_lower = url.lower()
    
    # Pattern 1: Contains 'podcast' keyword
    if 'podcast' in url_lower:
        return True
    
    # Pattern 2: Check if from podcast platform domains
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        
        # Remove www. prefix for matching
        domain = domain.replace('www.', '')
        
        # Podcast platform domains
        podcast_domains = [
            'podcasts.apple.com',
            'anchor.fm',
            'buzzsprout.com',
            'podbean.com',
            'libsyn.com',
            'stitcher.com',
        ]
        
        # Check exact domain match
        if domain in podcast_domains:
            return True
        
        # Check for podcast paths in common platforms
        path = parsed.path.lower()
        if domain in ['spotify.com', 'open.spotify.com'] and 'podcast' in path:
            return True
        if domain in ['soundcloud.com'] and 'podcast' in path:
            return True
        if domain in ['iheart.com', 'iheartradio.com'] and 'podcast' in path:
            return True
        if domain in ['audible.com'] and 'podcast' in path:
            return True
        
    except Exception as e:
        logger.warning(f"Error parsing URL {url}: {e}")
        return False
    
    return False


def contains_podcast_urls(urls: List[str]) -> Tuple[bool, List[str]]:
    """
    Check if any URL in the list is podcast-related.
    
    Returns:
        Tuple of (has_podcast_urls, list_of_podcast_urls_found)
    """
    podcast_urls = []
    for url in urls:
        if is_podcast_url(url):
            podcast_urls.append(url)
    
    return (len(podcast_urls) > 0, podcast_urls)


In [4]:
def process_file(file_path: Path) -> Tuple[bool, List[str], str]:
    """
    Process a single JSON file to check for podcast URLs.
    
    Returns:
        Tuple of (has_podcast_urls, podcast_urls_found, error_message)
        error_message is empty string if no error
    """
    try:
        # Load JSON file
        with open(file_path, 'r', encoding='utf-8') as f:
            urls = json.load(f)
        
        # Validate it's a list
        if not isinstance(urls, list):
            return (False, [], f"File does not contain a JSON array")
        
        # Check for podcast URLs
        has_podcast, podcast_urls = contains_podcast_urls(urls)
        
        return (has_podcast, podcast_urls, "")
        
    except json.JSONDecodeError as e:
        return (False, [], f"JSON decode error: {e}")
    except Exception as e:
        return (False, [], f"Error processing file: {e}")


In [5]:
def filter_and_move_podcast_files(source_folder: str, destination_folder: str):
    """
    Main function to filter JSON files containing podcast URLs and move them.
    
    Args:
        source_folder: Path to folder containing JSON files
        destination_folder: Path to folder where matching files will be moved
    """
    source_path = Path(source_folder)
    dest_path = Path(destination_folder)
    
    # Ensure destination exists
    dest_path.mkdir(parents=True, exist_ok=True)
    
    # Statistics
    stats = {
        'total_files': 0,
        'files_with_podcasts': 0,
        'files_moved': 0,
        'files_failed': 0,
        'total_podcast_urls_found': 0,
        'errors': []
    }
    
    # Sample podcast URLs found
    sample_podcast_urls = []
    
    logger.info("=" * 60)
    logger.info("Starting podcast URL filtering and file moving")
    logger.info(f"Source folder: {source_path}")
    logger.info(f"Destination folder: {dest_path}")
    logger.info("=" * 60)
    
    # Get all JSON files
    json_files = list(source_path.glob("*.json"))
    stats['total_files'] = len(json_files)
    
    logger.info(f"Found {stats['total_files']} JSON files to process")
    
    # Process each file
    for file_path in json_files:
        logger.info(f"Processing: {file_path.name}")
        
        has_podcast, podcast_urls, error = process_file(file_path)
        
        if error:
            stats['files_failed'] += 1
            stats['errors'].append(f"{file_path.name}: {error}")
            logger.warning(f"Error processing {file_path.name}: {error}")
            continue
        
        if has_podcast:
            stats['files_with_podcasts'] += 1
            stats['total_podcast_urls_found'] += len(podcast_urls)
            
            # Add to sample (keep first 10)
            if len(sample_podcast_urls) < 10:
                sample_podcast_urls.extend(podcast_urls[:10 - len(sample_podcast_urls)])
            
            # Move file to destination
            try:
                dest_file_path = dest_path / file_path.name
                shutil.move(str(file_path), str(dest_file_path))
                stats['files_moved'] += 1
                logger.info(f"✓ Moved {file_path.name} ({len(podcast_urls)} podcast URLs found)")
            except Exception as e:
                stats['files_failed'] += 1
                error_msg = f"{file_path.name}: Failed to move - {e}"
                stats['errors'].append(error_msg)
                logger.error(error_msg)
        else:
            logger.debug(f"  No podcast URLs found in {file_path.name}")
    
    # Print statistics
    logger.info("=" * 60)
    logger.info("Processing Complete!")
    logger.info(f"Total files scanned: {stats['total_files']}")
    logger.info(f"Files containing podcast URLs: {stats['files_with_podcasts']}")
    logger.info(f"Files successfully moved: {stats['files_moved']}")
    logger.info(f"Files failed to process/move: {stats['files_failed']}")
    logger.info(f"Total podcast URLs discovered: {stats['total_podcast_urls_found']}")
    
    if sample_podcast_urls:
        logger.info("\nSample podcast URLs found:")
        for url in sample_podcast_urls[:10]:
            logger.info(f"  - {url}")
    
    if stats['errors']:
        logger.warning(f"\nErrors encountered ({len(stats['errors'])}):")
        for error in stats['errors'][:10]:  # Show first 10 errors
            logger.warning(f"  - {error}")
        if len(stats['errors']) > 10:
            logger.warning(f"  ... and {len(stats['errors']) - 10} more errors")
    
    logger.info("=" * 60)
    
    return stats


In [6]:
# Execute the filtering and moving process
stats = filter_and_move_podcast_files(
    source_folder=SOURCE_FOLDER,
    destination_folder=DESTINATION_FOLDER
)


2025-12-22 12:51:27,371 - INFO - ============================================================
2025-12-22 12:51:27,372 - INFO - Starting podcast URL filtering and file moving
2025-12-22 12:51:27,373 - INFO - Source folder: apollo_podcast
2025-12-22 12:51:27,373 - INFO - Destination folder: apollo_podcast_urls
2025-12-22 12:51:27,374 - INFO - ============================================================
2025-12-22 12:51:27,474 - INFO - Found 22249 JSON files to process
2025-12-22 12:51:27,475 - INFO - Processing: 6d562bc629bed0df696084fa6f60925d.json
2025-12-22 12:51:27,476 - INFO - Processing: e86bacea85f31edf956ede106f194974.json
2025-12-22 12:51:27,477 - INFO - Processing: 3a63e967dcaeb1dcea05205688e7083f.json
2025-12-22 12:51:27,491 - INFO - Processing: 1056221c5b71b3cae72a1da62da43d39.json
2025-12-22 12:51:27,495 - INFO - ✓ Moved 1056221c5b71b3cae72a1da62da43d39.json (1 podcast URLs found)
2025-12-22 12:51:27,495 - INFO - Processing: c33af497a610b02b919fb8e98ae2478b.json
2025-12-22 1